# 02 · Parallelization with LangChain

When chain steps are **independent**, running them sequentially wastes time. `RunnableParallel` fans them out so they execute concurrently, then merges the results into a single dict.

This notebook covers three uses:
1. **Fan-out / fan-in** — generate several independent things at once, then combine them.
2. **Compare prompts/models** — run the same input through different prompts side by side.
3. **Timing proof** — measure parallel vs. sequential latency.

---

## Setup

In [1]:
%pip install -qU langchain langchain-openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel
from dotenv import load_dotenv

load_dotenv("../.env")

# Model comes from .env in "provider:model" form (e.g. openai:gpt-4o-mini),
# so switching models is a config change, not a code change.
llm = init_chat_model(os.environ["CHAT_MODEL"], temperature=0.7)
parser = StrOutputParser()

---
## 1. Fan-out / fan-in

Given a product idea, generate a **name**, a **tagline**, and a **target audience** at the same time — none of them depends on the others.

In [3]:
name_chain = (
    ChatPromptTemplate.from_template(
        "Suggest ONE catchy product name for: {idea}. Return only the name."
    ) | llm | parser
)

tagline_chain = (
    ChatPromptTemplate.from_template(
        "Write ONE punchy tagline (max 8 words) for: {idea}. Return only the tagline."
    ) | llm | parser
)

audience_chain = (
    ChatPromptTemplate.from_template(
        "Describe the ideal target audience for: {idea} in one sentence."
    ) | llm | parser
)

# Run all three concurrently. The dict keys become the output keys.
branding = RunnableParallel(
    name=name_chain,
    tagline=tagline_chain,
    audience=audience_chain,
)

In [4]:
result = branding.invoke({"idea": "a habit tracker that gamifies your mornings"})

for key, value in result.items():
    print(f"{key.upper():<10} {value}")

NAME       MorningQuest
TAGLINE    Win Your Mornings, Level Up Daily!
AUDIENCE   The ideal target audience for a habit tracker that gamifies your mornings is motivated individuals seeking to boost their daily productivity and motivation through engaging, playful routines.


> A plain `dict` of runnables is auto-coerced to `RunnableParallel`, so inside a pipe you can just write `{"name": name_chain, ...}`.

---
## 2. Parallel + a synthesis step (fan-in)

Run the branches in parallel, then pipe the merged dict into a final LLM call that combines them into a launch blurb.

In [5]:
synthesis_prompt = ChatPromptTemplate.from_template(
    "Write a 2-sentence product launch blurb.\n"
    "Name: {name}\nTagline: {tagline}\nAudience: {audience}"
)

# branding (parallel) -> synthesis (sequential)
launch_chain = branding | synthesis_prompt | llm | parser

print(launch_chain.invoke({"idea": "a habit tracker that gamifies your mornings"}))

Kickstart your day with MorningQuest, the gamified habit tracker designed to make building positive routines fun and rewarding. Win your mornings, level up your day—and transform your mornings into an epic adventure!


---
## 3. Compare prompts (or models) side by side

Same input, different system personas — run them together and compare.

In [6]:
def persona_chain(persona: str):
    return (
        ChatPromptTemplate.from_messages([
            ("system", f"You are {persona}. Answer in 2 sentences."),
            ("human", "{question}"),
        ]) | llm | parser
    )

compare = RunnableParallel(
    optimist=persona_chain("a relentless optimist"),
    skeptic=persona_chain("a hard-nosed skeptic"),
    pragmatist=persona_chain("a no-nonsense pragmatist"),
)

answers = compare.invoke({"question": "Should a startup adopt AI agents in 2026?"})
for persona, ans in answers.items():
    print(f"\n[{persona.upper()}]\n{ans}")


[OPTIMIST]
Absolutely! Embracing AI agents in 2026 will unlock innovative opportunities, enhance efficiency, and give your startup a competitive edge in a rapidly evolving landscape. The future belongs to those who dare to innovate now!

[SKEPTIC]
Adopting AI agents in 2026 can offer competitive advantages, but startups should carefully consider the costs, integration challenges, and potential ethical issues before rushing in. Without thorough planning and clear ROI, jumping into AI adoption might do more harm than good.

[PRAGMATIST]
Yes, adopting AI agents in 2026 can provide a competitive edge, streamline operations, and enhance decision-making, but it should be done with clear goals and risk management in mind. Evaluate the specific needs and resources of your startup to ensure the investment aligns with your growth strategy.


---
## 4. Proof: parallel is faster

Compare wall-clock time for running the three branding branches concurrently vs. one after another.

In [7]:
import time

idea = {"idea": "a habit tracker that gamifies your mornings"}

# Parallel
t0 = time.perf_counter()
branding.invoke(idea)
parallel_t = time.perf_counter() - t0

# Sequential (invoke each branch one at a time)
t0 = time.perf_counter()
for chain in (name_chain, tagline_chain, audience_chain):
    chain.invoke(idea)
sequential_t = time.perf_counter() - t0

print(f"Parallel:   {parallel_t:.2f}s")
print(f"Sequential: {sequential_t:.2f}s")
print(f"Speedup:    {sequential_t / parallel_t:.1f}x")

Parallel:   1.24s
Sequential: 2.33s
Speedup:    1.9x


---
## What to remember

| Concept | What it does |
|---|---|
| `RunnableParallel(a=..., b=...)` | Runs branches concurrently, returns a dict keyed by name |
| `{"a": chain_a, "b": chain_b}` | Plain dict auto-coerces to `RunnableParallel` inside a pipe |
| `parallel | next_step` | Fan-out then fan-in — merge results into a synthesis step |
| `.batch([...])` | The other axis: same chain over many inputs concurrently |

**Next:** Article 03 — **Routing**: send each input to a *different* chain based on its content.